# TASI Schema & Data Validation

## Overview

This notebook validates the cleaned TASI dataset before it is used in downstream integration and analytics.

The validation process checks:

- Dataset schema and column structure.
- Expected data types.
- Missing values and duplicate records.
- Valid price and volume values.
- Date ordering and coverage.
- Logical relationships between market price fields.
- Source-level data quality observations.

The validated dataset will be used as an input for the integration stage.

In [4]:
# Step 1 — Load the cleaned TASI dataset

import pandas as pd

processed_path = "../data/processed/tasi_cleaned.csv"

df = pd.read_csv(
    processed_path,
    parse_dates=["Date"]
)

print("===== CLEANED DATA LOADED =====")
print("Shape:", df.shape)

display(df.head())

===== CLEANED DATA LOADED =====
Shape: (3174, 7)


,Date,Close,Open,High,Low,Volume,Change%
0,2014-01-01,8605.34,8535.60,8605.56,8535.22,184630000.0,0.82
1,2014-01-02,8618.12,8605.34,8621.49,8572.34,176630000.0,0.15
2,2014-01-05,8637.74,8618.12,8638.25,8602.35,184300000.0,0.23
3,2014-01-06,8611.81,8637.74,8637.74,8594.78,204100000.0,-0.30
4,2014-01-07,8608.80,8611.81,8620.37,8588.05,205950000.0,-0.03


## Step 2 — Validate Dataset Schema

The cleaned TASI dataset is checked against the expected schema.

The validation ensures that:

- All required columns are present.
- No unexpected columns exist.
- The column order matches the expected structure.

Expected schema:

`Date`, `Close`, `Open`, `High`, `Low`, `Volume`, `Change%`

In [5]:
# Step 2 — Validate the dataset schema

expected_columns = [
    "Date",
    "Close",
    "Open",
    "High",
    "Low",
    "Volume",
    "Change%"
]

actual_columns = df.columns.tolist()

print("===== SCHEMA VALIDATION =====")

print("\nExpected Columns:")
print(expected_columns)

print("\nActual Columns:")
print(actual_columns)

print("\nAll Required Columns Present:")
print(set(expected_columns).issubset(set(actual_columns)))

print("\nNo Unexpected Columns:")
print(set(actual_columns).issubset(set(expected_columns)))

print("\nColumn Order Correct:")
print(actual_columns == expected_columns)

===== SCHEMA VALIDATION =====

Expected Columns:
['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Change%']

Actual Columns:
['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Change%']

All Required Columns Present:
True

No Unexpected Columns:
True

Column Order Correct:
True


## Step 3 — Validate Data Types

The data types of the cleaned dataset are compared with the expected data types.

Expected types:

- `Date` → datetime
- `Close`, `Open`, `High`, `Low` → numeric
- `Volume` → numeric
- `Change%` → numeric

This ensures that the dataset is stored in a format suitable for downstream processing and analytics.

In [7]:
# Step 3 — Validate data types

expected_dtypes = {
    "Date": "datetime",
    "Close": "numeric",
    "Open": "numeric",
    "High": "numeric",
    "Low": "numeric",
    "Volume": "numeric",
    "Change%": "numeric"
}

print("===== DATA TYPE VALIDATION =====")

for column, expected_type in expected_dtypes.items():
    actual_type = str(df[column].dtype)

    if column == "Date":
        is_valid = pd.api.types.is_datetime64_any_dtype(df[column])
    else:
        is_valid = pd.api.types.is_numeric_dtype(df[column])

    print(
        f"{column}: {actual_type} | "
        f"Expected: {expected_type} | "
        f"Valid: {is_valid}"
    )

===== DATA TYPE VALIDATION =====
Date: datetime64[us] | Expected: datetime | Valid: True
Close: float64 | Expected: numeric | Valid: True
Open: float64 | Expected: numeric | Valid: True
High: float64 | Expected: numeric | Valid: True
Low: float64 | Expected: numeric | Valid: True
Volume: float64 | Expected: numeric | Valid: True
Change%: float64 | Expected: numeric | Valid: True


## Step 4 — Completeness & Duplicate Validation

The cleaned dataset is checked for completeness and duplicate records.

The validation confirms that:

- No required fields contain missing values.
- No duplicate records are present.

This ensures that the dataset is complete and does not contain repeated observations before further validation.

In [8]:
# Step 4 — Validate completeness and duplicates

print("===== COMPLETENESS & DUPLICATE VALIDATION =====")

# Missing values
missing_values = df.isna().sum()

print("\nMissing Values by Column:")
display(missing_values.to_frame("Missing Values"))

print("\nTotal Missing Values:")
print(missing_values.sum())

# Duplicate rows
duplicate_rows = df.duplicated().sum()

print("\nDuplicate Rows:")
print(duplicate_rows)

# Overall validation result
print("\nCompleteness Check Passed:")
print(missing_values.sum() == 0)

print("\nDuplicate Check Passed:")
print(duplicate_rows == 0)

===== COMPLETENESS & DUPLICATE VALIDATION =====

Missing Values by Column:


,Missing Values
Date,0
Close,0
Open,0
High,0
Low,0
Volume,0
Change%,0



Total Missing Values:
0

Duplicate Rows:
0

Completeness Check Passed:
True

Duplicate Check Passed:
True


## Step 5 — Market Data Rules Validation

The cleaned TASI dataset is checked against basic market data rules to identify invalid observations.

The validation checks:

- Prices must be greater than zero.
- Trading volume must not be negative.
- `High` must be greater than or equal to `Low`.
- Dates must be in ascending order.

These rules help identify records that could affect the reliability of downstream analysis.

In [9]:
# Step 5 — Validate market data rules

price_columns = ["Close", "Open", "High", "Low"]

print("===== MARKET DATA RULE VALIDATION =====")

# 1. Non-positive prices
non_positive_prices = (df[price_columns] <= 0).sum().sum()

print("\n1. Non-positive Price Values:")
print(non_positive_prices)

# 2. Negative volume
negative_volume = (df["Volume"] < 0).sum()

print("\n2. Negative Volume Values:")
print(negative_volume)

# 3. Invalid High-Low relationship
invalid_high_low = (df["High"] < df["Low"]).sum()

print("\n3. High < Low:")
print(invalid_high_low)

# 4. Date ordering
dates_sorted = df["Date"].is_monotonic_increasing

print("\n4. Dates Sorted Ascending:")
print(dates_sorted)

# Overall result
print("\n===== VALIDATION SUMMARY =====")
print("Price Check Passed:", non_positive_prices == 0)
print("Volume Check Passed:", negative_volume == 0)
print("High-Low Check Passed:", invalid_high_low == 0)
print("Date Order Check Passed:", dates_sorted)

===== MARKET DATA RULE VALIDATION =====

1. Non-positive Price Values:
0

2. Negative Volume Values:
0

3. High < Low:
0

4. Dates Sorted Ascending:
True

===== VALIDATION SUMMARY =====
Price Check Passed: True
Volume Check Passed: True
High-Low Check Passed: True
Date Order Check Passed: True


## Step 6 — Investigate Open Price Anomalies

The `Open` price is expected to fall within the same day's `High-Low` range under a conventional OHLC interpretation.

However, the source dataset contains records where `Open` falls outside this range.

Instead of removing these records automatically, the observations are investigated by comparing each `Open` value with the previous trading day's `Close`.

This helps determine whether the records represent a data-quality issue or a source-level market data behavior.

In [10]:
# Step 6 — Investigate Open price anomalies

# Identify records where Open falls outside the same day's High-Low range
open_outside_range = (
    (df["Open"] > df["High"]) |
    (df["Open"] < df["Low"])
)

anomaly_df = df.loc[
    open_outside_range,
    ["Date", "Open", "High", "Low", "Close"]
].copy()

print("===== OPEN PRICE ANOMALY INVESTIGATION =====")

print("\nRecords with Open outside High-Low range:")
print(len(anomaly_df))

# Compare with the previous trading day's Close
df["Previous_Close"] = df["Close"].shift(1)

anomaly_df["Previous_Close"] = df.loc[
    open_outside_range,
    "Previous_Close"
].values

# Check whether Open equals the previous trading day's Close
anomaly_df["Matches_Previous_Close"] = (
    anomaly_df["Open"] == anomaly_df["Previous_Close"]
)

print("\nAnomalies matching Previous Trading Day Close:")
print(anomaly_df["Matches_Previous_Close"].sum())

print("\nTotal anomalies with a Previous Close:")
print(anomaly_df["Previous_Close"].notna().sum())

display(anomaly_df.head())

===== OPEN PRICE ANOMALY INVESTIGATION =====

Records with Open outside High-Low range:
300

Anomalies matching Previous Trading Day Close:
300

Total anomalies with a Previous Close:
300


,Date,Open,High,Low,Close,Previous_Close,Matches_Previous_Close
431,2015-09-20,7470.19,7469.46,7343.35,7365.98,7470.19,True
432,2015-09-21,7365.98,7443.79,7368.40,7442.71,7365.98,True
446,2015-10-18,7698.73,7800.10,7699.53,7792.62,7698.73,True
466,2015-11-15,7083.43,7082.62,6860.59,6881.42,7083.43,True
470,2015-11-19,6953.47,7051.98,6954.27,7034.08,6953.47,True


## Step 7 — Validation Findings

The validation identified 300 records where the `Open` price falls outside the same day's `High-Low` range.

Further investigation showed that all 300 records have an `Open` value equal to the previous trading day's `Close`.

Therefore, these records are retained in the dataset. No rows are removed based on this observation.

This finding is documented as a source-level market data behavior and does not affect the completeness or structural validity of the dataset.

In [11]:
# Step 7 — Remove the temporary investigation column

if "Previous_Close" in df.columns:
    df.drop(columns=["Previous_Close"], inplace=True)

print("===== TEMPORARY COLUMN REMOVED =====")
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)

===== TEMPORARY COLUMN REMOVED =====
Columns: ['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Change%']
Shape: (3174, 7)


## Step 8 — Final Validation Summary

The TASI dataset has completed schema and data-quality validation.

The final validation confirms:

- The expected schema is present.
- All data types are valid.
- No missing values are present.
- No duplicate records are present.
- All prices are positive.
- No negative volume values exist.
- `High` is not lower than `Low`.
- Dates are sorted chronologically.
- The 300 Open-price observations identified during investigation were retained because they matched the previous trading day's Close.

The validated dataset contains 3,174 records and 7 columns and is ready for downstream data integration.

In [12]:
# Step 8 — Final validation summary

print("===== FINAL VALIDATION SUMMARY =====")

print("\nDataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nTotal Missing Values:")
print(df.isna().sum().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nDate Range:")
print("First Date:", df["Date"].min())
print("Last Date:", df["Date"].max())

print("\nFinal Validation Status:")

schema_valid = df.columns.tolist() == [
    "Date", "Close", "Open", "High", "Low", "Volume", "Change%"
]

dtypes_valid = (
    pd.api.types.is_datetime64_any_dtype(df["Date"])
    and all(
        pd.api.types.is_numeric_dtype(df[col])
        for col in ["Close", "Open", "High", "Low", "Volume", "Change%"]
    )
)

completeness_valid = df.isna().sum().sum() == 0
duplicates_valid = df.duplicated().sum() == 0
prices_valid = (df[["Close", "Open", "High", "Low"]] > 0).all().all()
volume_valid = (df["Volume"] >= 0).all()
high_low_valid = (df["High"] >= df["Low"]).all()
dates_valid = df["Date"].is_monotonic_increasing

print("Schema:", schema_valid)
print("Data Types:", dtypes_valid)
print("Completeness:", completeness_valid)
print("Duplicates:", duplicates_valid)
print("Prices:", prices_valid)
print("Volume:", volume_valid)
print("High-Low:", high_low_valid)
print("Date Order:", dates_valid)

final_status = all([
    schema_valid,
    dtypes_valid,
    completeness_valid,
    duplicates_valid,
    prices_valid,
    volume_valid,
    high_low_valid,
    dates_valid
])

print("\nOverall Validation Status:", final_status)

===== FINAL VALIDATION SUMMARY =====

Dataset Shape:
(3174, 7)

Columns:
['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Change%']

Total Missing Values:
0

Duplicate Rows:
0

Date Range:
First Date: 2014-01-01 00:00:00
Last Date: 2026-09-20 00:00:00

Final Validation Status:
Schema: True
Data Types: True
Completeness: True
Duplicates: True
Prices: True
Volume: True
High-Low: True
Date Order: True

Overall Validation Status: True


In [13]:
# Step 9 — Confirm the validated processed dataset

from pathlib import Path

processed_path = Path("../data/processed/tasi_cleaned.csv")

print("===== VALIDATED DATASET CONFIRMATION =====")

print("File:", processed_path.resolve())
print("File Exists:", processed_path.exists())
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Status: Validated processed dataset is ready for downstream integration.")

===== VALIDATED DATASET CONFIRMATION =====
File: C:\Users\aldos\Saudi_FinHub_Project\data\processed\tasi_cleaned.csv
File Exists: True
Rows: 3174
Columns: 7
Status: Validated processed dataset is ready for downstream integration.
